# School Intelligence Database — Query Notebook

Search and ask questions about crawled school data stored in Qdrant.

## 1. Setup

In [ ]:
import os, sys
from dotenv import load_dotenv
load_dotenv()

from qdrant_client import QdrantClient
from qdrant_client.models import PayloadSchemaType
from embedder import get_embedder

# Connect to Qdrant
client = QdrantClient(
    url=os.getenv('QDRANT_URL'),
    api_key=os.getenv('QDRANT_API_KEY')
)
COLLECTION     = os.getenv('QDRANT_COLLECTION', 'school_intel')
RAW_COLLECTION = os.getenv('QDRANT_RAW_COLLECTION', 'school_intel_raw')

# Ensure payload indexes exist for filtered fields
for col in [COLLECTION, RAW_COLLECTION]:
    try:
        for field in ['metadata.type', 'metadata.school_name']:
            client.create_payload_index(
                collection_name=col,
                field_name=field,
                field_schema=PayloadSchemaType.KEYWORD,
            )
    except Exception:
        pass  # collection may not exist yet

# Load embedder
embedder = get_embedder()

print(f"Connected to Qdrant")
print(f"  Entity collection: {COLLECTION} ({client.count(COLLECTION).count} points)")
try:
    raw_count = client.count(RAW_COLLECTION).count
    print(f"  Raw text collection: {RAW_COLLECTION} ({raw_count} points)")
except Exception:
    print(f"  Raw text collection: {RAW_COLLECTION} (not created yet)")
print("Payload indexes ensured")

2026-03-01 23:24:20.742 | INFO     | embedder:__init__:119 - Loading HuggingFace model: BAAI/bge-large-en-v1.5 on cuda...
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 909.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-01 23:24:25.931 | INFO     | embedder:__init__:123 - HuggingFace embedder ready | BAAI/bge-large-en-v1.5 | cuda | dim=1024


Connected to Qdrant | Collection: school_intel
Total points: 363
Payload indexes ensured for metadata.type, metadata.school_name


## 2. Semantic Search

Ask any question — it will find the most relevant chunks.

In [19]:
def search(query: str, limit: int = 10, entity_type: str = None, school: str = None):
    """Search Qdrant with optional filters."""
    from qdrant_client.models import Filter, FieldCondition, MatchValue

    # Embed the query
    [query_vector] = embedder.embed([query])

    # Build filters
    conditions = []
    if entity_type:
        conditions.append(FieldCondition(key='metadata.type', match=MatchValue(value=entity_type)))
    if school:
        conditions.append(FieldCondition(key='metadata.school_name', match=MatchValue(value=school)))

    qdrant_filter = Filter(must=conditions) if conditions else None

    response = client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        query_filter=qdrant_filter,
        limit=limit,
        with_payload=True,
    )
    results = response.points

    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    if entity_type: print(f"Filter: type={entity_type}")
    if school: print(f"Filter: school={school}")
    print(f"Results: {len(results)}")
    print(f"{'='*70}")

    for i, hit in enumerate(results, 1):
        payload = hit.payload
        meta = payload.get('metadata', {})
        print(f"\n[{i}] Score: {hit.score:.4f} | {meta.get('type', '').upper()}")
        print(f"    Text: {payload.get('page_content', '')[:150]}")
        print(f"    Source: {meta.get('source_url', '')}")
        print(f"    School: {meta.get('school_name', '')}")

    return results

In [20]:
# Try it — change the query to whatever you want
results = search("board members and leadership")


Query: 'board members and leadership'
Results: 10

[1] Score: 0.6622 | BOARD_MEMBER
    Text: Board member: . School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/pages/index.jsp?uREC_ID=1323100&type=u
    School: Aceacademycharter

[2] Score: 0.6600 | BOARD_MEMBER
    Text: Board member: Bridgette Reese. Role: Principal. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/pages/index.jsp?uREC_ID=471369&type=d
    School: Aceacademycharter

[3] Score: 0.6544 | BOARD_MEMBER
    Text: Board member: Mrs. Hammond. Role: after-school Director. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/pages/index.jsp?uREC_ID=471332&type=d
    School: Aceacademycharter

[4] Score: 0.6532 | BOARD_MEMBER
    Text: Board member: Dr. Bridgette Reese. Role: Principal. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/staff/
    School: Aceacademycharter

[5] Score: 0.6447 | BOARD_MEMBER
    Text: Board 

In [21]:
# Filter by entity type: vendor, budget, project, problem, board_member, contractor
results = search("contracts", entity_type="vendor")


Query: 'contracts'
Filter: type=vendor
Results: 10

[1] Score: 0.6588 | VENDOR
    Text: Vendor: Google. Service: educational platform. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/ourpages/auto/2025/7/11/54551642/5th%20Grade%20School%20Supply%20List.pdf
    School: Aceacademycharter

[2] Score: 0.6588 | VENDOR
    Text: Vendor: Google. Service: educational platform. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/ourpages/auto/2025/7/11/54551642/Kindergarten%20School%20Supply%2025-26.pdf
    School: Aceacademycharter

[3] Score: 0.6428 | VENDOR
    Text: Vendor: Google. Service: Workspace for Education. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/ourpages/auto/2025/7/11/54551642/6th%20Grade%20School%20Supply%20List%2025-26.pdf
    School: Aceacademycharter

[4] Score: 0.6428 | VENDOR
    Text: Vendor: GOOGLE. Service: WORKSPACE FOR EDUCATION. School: Aceacademycharter
    Source: https://www.aceacademychar

In [22]:
# Filter by school
results = search("budget", school="Aceacademycharter")


Query: 'budget'
Filter: school=Aceacademycharter
Results: 10

[1] Score: 0.7626 | BUDGET
    Text: Budget: food and essential goods. Amount: $ 5. Funded by: each person in every household. Status: asking. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/
    School: Aceacademycharter

[2] Score: 0.7393 | BUDGET
    Text: Budget: purchasing food and essential goods. Amount: USD 5. Funded by: donations. Status: fundraising. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0
    School: Aceacademycharter

[3] Score: 0.7286 | BUDGET
    Text: Budget: program fees. Funded by: parents. Period: monthly. Status: required. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/pages/index.jsp?uREC_ID=471332&type=d
    School: Aceacademycharter

[4] Score: 0.7274 | BUDGET
    Text: Budget: general. School: Aceacademycharter
    Source: https://www.aceacademycharter.org/apps/pages/index.j

## 2b. Raw Text Search

Search the raw page text collection for deeper context the entity extractor may have missed.

In [ ]:
def search_raw(query: str, limit: int = 10, school: str = None):
    """Search the raw text collection."""
    from qdrant_client.models import Filter, FieldCondition, MatchValue

    [query_vector] = embedder.embed([query])

    conditions = []
    if school:
        conditions.append(FieldCondition(key='metadata.school_name', match=MatchValue(value=school)))

    qdrant_filter = Filter(must=conditions) if conditions else None

    try:
        response = client.query_points(
            collection_name=RAW_COLLECTION,
            query=query_vector,
            query_filter=qdrant_filter,
            limit=limit,
            with_payload=True,
        )
        results = response.points
    except Exception as e:
        print(f"Raw collection not available: {e}")
        return []

    print(f"\n{'='*70}")
    print(f"Raw Text Search: '{query}'")
    if school: print(f"Filter: school={school}")
    print(f"Results: {len(results)}")
    print(f"{'='*70}")

    for i, hit in enumerate(results, 1):
        payload = hit.payload
        meta = payload.get('metadata', {})
        text = payload.get('page_content', '')
        print(f"\n[{i}] Score: {hit.score:.4f}")
        print(f"    Source: {meta.get('source_url', '')}")
        print(f"    Title: {meta.get('title', '')}")
        print(f"    Text: {text[:300]}...")

    return results

In [ ]:
# Search raw text — finds context the entity extractor missed
search_raw("vendors and services used by the school")

## 3. RAG — Ask Questions with Gemini

Retrieves relevant chunks, then asks Gemini to answer based on them.

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-2.5-flash')

def ask(question: str, limit: int = 10, entity_type: str = None, school: str = None,
        use_raw: bool = True):
    """
    RAG: retrieve context from BOTH entity + raw text collections, then answer with Gemini.

    use_raw=True (default) merges results from the raw text collection for deeper context.
    """
    from qdrant_client.models import Filter, FieldCondition, MatchValue

    [query_vector] = embedder.embed([question])

    conditions = []
    if entity_type:
        conditions.append(FieldCondition(key='metadata.type', match=MatchValue(value=entity_type)))
    if school:
        conditions.append(FieldCondition(key='metadata.school_name', match=MatchValue(value=school)))

    qdrant_filter = Filter(must=conditions) if conditions else None

    # Search entity collection
    entity_resp = client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        query_filter=qdrant_filter,
        limit=limit,
        with_payload=True,
    )
    entity_hits = entity_resp.points

    # Search raw text collection for deeper context
    raw_hits = []
    if use_raw:
        try:
            # Build filter without entity_type (raw chunks are all type=raw_text)
            raw_conditions = []
            if school:
                raw_conditions.append(FieldCondition(key='metadata.school_name', match=MatchValue(value=school)))
            raw_filter = Filter(must=raw_conditions) if raw_conditions else None

            raw_resp = client.query_points(
                collection_name=RAW_COLLECTION,
                query=query_vector,
                query_filter=raw_filter,
                limit=limit,
                with_payload=True,
            )
            raw_hits = raw_resp.points
        except Exception:
            pass  # raw collection may not exist yet

    all_hits = entity_hits + raw_hits
    if not all_hits:
        print("No relevant data found.")
        return

    # Build context — entity chunks first, then raw text for depth
    context_parts = []
    for i, hit in enumerate(entity_hits, 1):
        payload = hit.payload
        meta = payload.get('metadata', {})
        text = payload.get('page_content', '')
        source = meta.get('source_url', '')
        etype = meta.get('type', '')
        context_parts.append(f"[{i}] [{etype.upper()}] {text}\n    Source: {source}")

    offset = len(entity_hits)
    for i, hit in enumerate(raw_hits, offset + 1):
        payload = hit.payload
        meta = payload.get('metadata', {})
        text = payload.get('page_content', '')
        source = meta.get('source_url', '')
        context_parts.append(f"[{i}] [RAW PAGE TEXT] {text[:500]}\n    Source: {source}")

    context = '\n\n'.join(context_parts)

    prompt = f"""You are a school intelligence analyst. Answer the question using ONLY the context below.
Use ALL the context provided — both structured entity data and raw page text.
If the answer is not in the context, say so. Cite source numbers [1], [2] etc.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = model.generate_content(prompt)
    print(f"\n{'='*70}")
    print(f"Question: {question}")
    print(f"Sources: {len(entity_hits)} entity chunks + {len(raw_hits)} raw text chunks")
    print(f"{'='*70}\n")
    print(response.text)
    return response.text

In [24]:
# Ask a question
ask("Who are the board members and what are their roles?")


Question: Who are the board members and what are their roles?
Sources used: 10 chunks

The board members and their roles are:
*   Shawn Smalls: Board Chair [3]
*   Jessica Hales: Library/Media Coordinator/School Events [2]
*   Bridgette Reese: Principal [5], [9]
*   Shannon Martin: Director [6], [7]
*   Dawn Hammond: Director of Student & Family Services [10] (also identified as Mrs. Hammond, after-school Director [1], and Director [8])


'The board members and their roles are:\n*   Shawn Smalls: Board Chair [3]\n*   Jessica Hales: Library/Media Coordinator/School Events [2]\n*   Bridgette Reese: Principal [5], [9]\n*   Shannon Martin: Director [6], [7]\n*   Dawn Hammond: Director of Student & Family Services [10] (also identified as Mrs. Hammond, after-school Director [1], and Director [8])'

In [36]:
ask("What vendors does the school use and what is the usecase?")


Question: What vendors does the school use and what is the usecase?
Sources used: 10 chunks

The school, Aceacademycharter, uses the following vendors and their services:

*   **Google**: educational platform [1], [2], Workspace for Education [3], [4], [7]
*   **Amazon**: fundraising program [5]
*   **Infinite Campus**: student information system [6]
*   **GoFundMe**: fundraising platform [8], [9]
*   **Department of Agriculture (USDA)**: conducting or funding programs [10]


'The school, Aceacademycharter, uses the following vendors and their services:\n\n*   **Google**: educational platform [1], [2], Workspace for Education [3], [4], [7]\n*   **Amazon**: fundraising program [5]\n*   **Infinite Campus**: student information system [6]\n*   **GoFundMe**: fundraising platform [8], [9]\n*   **Department of Agriculture (USDA)**: conducting or funding programs [10]'

In [35]:
ask("tell me about  Project HOME ")


Question: tell me about  Project HOME 
Sources used: 10 chunks

Project H.O.M.E. (Hear Our Message Everyone: People Need A Home) is a project at Aceacademycharter. It involves writing a book filled with heartfelt stories, with each story aiming to shed light on the housing crisis and celebrate the strength of those overcoming it [1, 2].


'Project H.O.M.E. (Hear Our Message Everyone: People Need A Home) is a project at Aceacademycharter. It involves writing a book filled with heartfelt stories, with each story aiming to shed light on the housing crisis and celebrate the strength of those overcoming it [1, 2].'

In [27]:
ask("What problems or issues has the school reported?")


Question: What problems or issues has the school reported?
Sources used: 10 chunks

The school has reported the following problems or issues:
*   Burning of a school building [2]
*   Inappropriate Items on school property [3]
*   Assault on student [4]
*   Misuse of school technology [5]
*   Possession of marijuana [6]
*   Use of alcoholic beverages [7]
*   Inappropriate language [8]
*   Property damage [9]
*   Bus misbehavior [10]


'The school has reported the following problems or issues:\n*   Burning of a school building [2]\n*   Inappropriate Items on school property [3]\n*   Assault on student [4]\n*   Misuse of school technology [5]\n*   Possession of marijuana [6]\n*   Use of alcoholic beverages [7]\n*   Inappropriate language [8]\n*   Property damage [9]\n*   Bus misbehavior [10]'

## 4. Browse All Records for a School

In [28]:
def browse(school: str = None, entity_type: str = None, limit: int = 50):
    """Scroll through records without a query."""
    from qdrant_client.models import Filter, FieldCondition, MatchValue

    conditions = []
    if school:
        conditions.append(FieldCondition(key='metadata.school_name', match=MatchValue(value=school)))
    if entity_type:
        conditions.append(FieldCondition(key='metadata.type', match=MatchValue(value=entity_type)))

    scroll_filter = Filter(must=conditions) if conditions else None

    results, _ = client.scroll(
        collection_name=COLLECTION,
        scroll_filter=scroll_filter,
        limit=limit,
        with_payload=True,
    )

    print(f"Found {len(results)} records")
    for i, point in enumerate(results, 1):
        meta = point.payload.get('metadata', {})
        print(f"\n[{i}] {meta.get('type', '').upper()} | {point.payload.get('page_content', '')[:120]}")

    return results

In [29]:
# Browse all vendors
browse(entity_type="vendor")

Found 50 records

[1] VENDOR | Vendor: BlockLoveCLT. School: Aceacademycharter

[2] VENDOR | Vendor: Amazon Smile. School: Aceacademycharter

[3] VENDOR | Vendor: Google Workspace. Service: platform includes tools such as Gmail, Google Docs, Google Classroom, Calendar. Schoo

[4] VENDOR | Vendor: Google. Service: educational platform. School: Aceacademycharter

[5] VENDOR | Vendor: Amazon. Service: Smile program. School: Aceacademycharter

[6] VENDOR | Vendor: Google. Service: educational platform. School: Aceacademycharter

[7] VENDOR | Vendor: Infinite Campus. School: Aceacademycharter

[8] VENDOR | Vendor: Amazon Smile. School: Aceacademycharter

[9] VENDOR | Vendor: Ziplock. Service: bags. School: Aceacademycharter

[10] VENDOR | Vendor: Asociación de Lesiones Cerebrales de Carolina del Norte. School: Aceacademycharter

[11] VENDOR | Vendor: A.C.E. Academy Preschool. Service: smooth transition into kindergarten and beyond. School: Aceacademycharter

[12] VENDOR | Vendor: Amazon Smi

[Record(id='00267758-32f4-a750-ba3f-e07a49c10c44', payload={'page_content': 'Vendor: BlockLoveCLT. School: Aceacademycharter', 'metadata': {'school_name': 'Aceacademycharter', 'domain': 'www.aceacademycharter.org', 'type': 'vendor', 'source': 'https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0', 'source_type': 'website', 'source_page': None, 'raw_text': 'BlockLoveCLT', 'vendor_name': 'BlockLoveCLT', 'service_type': '', 'contract_value': '', 'expiry_date': '', 'status': '', 'source_url': 'https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0', 'source_domain': 'www.aceacademycharter.org', 'source_school_name': 'Aceacademycharter', 'source_filename': None, 'source_is_pdf': False, 'source_crawled_at': '2026-03-01T17:50:56.493522+00:00', 'source_chunk_text': 'BlockLoveCLT', 'source_label': 'Aceacademycharter — Website (https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0)'}}, vector=None, shard_key=None, order_value=N

In [30]:
# Browse all records for a specific school
browse(school="Aceacademycharter")

Found 50 records

[1] VENDOR | Vendor: BlockLoveCLT. School: Aceacademycharter

[2] BUDGET | Budget: program fees. Funded by: parents. Period: monthly. Status: required. School: Aceacademycharter

[3] VENDOR | Vendor: Amazon Smile. School: Aceacademycharter

[4] PROBLEM | Problem: Assault on non-student w/o weapon. Category: safety. Severity: high. School: Aceacademycharter

[5] PROBLEM | Problem: Concusiones cerebrales y retorno a la actividad. Category: Salud y Seguridad. Severity: Grave. Resolution: perm

[6] VENDOR | Vendor: Google Workspace. Service: platform includes tools such as Gmail, Google Docs, Google Classroom, Calendar. Schoo

[7] PROBLEM | Problem: Bomb Threat. Category: threat. Severity: high. School: Aceacademycharter

[8] PROBLEM | Problem: concusión. Category: health. Severity: serious. Resolution: No debes volver a jugar o practicar el mismo día qu

[9] BUDGET | Budget: purchasing food and essential goods. Amount: USD 5. Funded by: donations. Status: fundraising. Sc

[Record(id='00267758-32f4-a750-ba3f-e07a49c10c44', payload={'page_content': 'Vendor: BlockLoveCLT. School: Aceacademycharter', 'metadata': {'school_name': 'Aceacademycharter', 'domain': 'www.aceacademycharter.org', 'type': 'vendor', 'source': 'https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0', 'source_type': 'website', 'source_page': None, 'raw_text': 'BlockLoveCLT', 'vendor_name': 'BlockLoveCLT', 'service_type': '', 'contract_value': '', 'expiry_date': '', 'status': '', 'source_url': 'https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0', 'source_domain': 'www.aceacademycharter.org', 'source_school_name': 'Aceacademycharter', 'source_filename': None, 'source_is_pdf': False, 'source_crawled_at': '2026-03-01T17:50:56.493522+00:00', 'source_chunk_text': 'BlockLoveCLT', 'source_label': 'Aceacademycharter — Website (https://www.aceacademycharter.org/apps/news/show_news.jsp?REC_ID=993368&id=0)'}}, vector=None, shard_key=None, order_value=N

## 5. Stats

In [ ]:
from collections import Counter

# Entity collection stats
all_points, _ = client.scroll(COLLECTION, limit=10000, with_payload=True)

types = Counter(p.payload.get('metadata', {}).get('type', 'unknown') for p in all_points)
schools = Counter(p.payload.get('metadata', {}).get('school_name', 'unknown') for p in all_points)

print(f"=== Entity Collection: {COLLECTION} ===")
print(f"Total records: {len(all_points)}")
print(f"\nBy entity type:")
for t, n in types.most_common():
    print(f"  {t:20s} {n}")
print(f"\nBy school:")
for s, n in schools.most_common():
    print(f"  {s:30s} {n}")

# Raw text collection stats
try:
    raw_points, _ = client.scroll(RAW_COLLECTION, limit=10000, with_payload=True)
    raw_schools = Counter(p.payload.get('metadata', {}).get('school_name', 'unknown') for p in raw_points)
    print(f"\n=== Raw Text Collection: {RAW_COLLECTION} ===")
    print(f"Total raw text chunks: {len(raw_points)}")
    print(f"\nBy school:")
    for s, n in raw_schools.most_common():
        print(f"  {s:30s} {n}")
except Exception:
    print(f"\nRaw text collection '{RAW_COLLECTION}' not created yet.")

Total records: 363

By entity type:
  problem              168
  vendor               89
  project              46
  budget               40
  board_member         20

By school:
  Aceacademycharter              363
